# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanaanwar25/flyrank-ml-internship-sana/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule prioritizes pages with low recent sessions but meaningful search demand. Pages with stronger search demand and weaker sessions receive higher review priority. The rule is a simple decision-support baseline, not a prediction of future performance.

Reason codes:
- HIGH_DEMAND_LOW_SESSIONS: relatively strong search demand with low sessions.
- MODERATE_DEMAND_LOW_SESSIONS: moderate search demand with low sessions.
- OTHER: does not meet the priority conditions.

In [16]:
import os
import pandas as pd

repo = "/content/flyrank-ml-internship-starter"

# Clone the repository if it is not already present
if not os.path.exists(repo):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

# Move into the repository
os.chdir(repo)

# Load the data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline score gives priority to pages with higher search demand and lower recent sessions. I rank the pages from highest to lowest score and save the resulting queue as a CSV for review.

In [17]:
import os

# Re-create the reason code
session_cutoff = df["sessions_90d"].median()
search_cutoff = df["search_volume"].median()

def reason_code(row):
    if row["search_volume"] >= search_cutoff and row["sessions_90d"] < session_cutoff:
        return "HIGH_DEMAND_LOW_SESSIONS"
    elif row["sessions_90d"] < session_cutoff:
        return "MODERATE_DEMAND_LOW_SESSIONS"
    else:
        return "OTHER"

df["reason_code"] = df.apply(reason_code, axis=1)

# Create score and ranking
df["baseline_action_score"] = (
    df["search_volume"].rank(pct=True)
    + (1 - df["sessions_90d"].rank(pct=True))
)

df = df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)

# Save
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df))
print(df[["rank", "baseline_action_score", "reason_code"]].head(10))

Saved: work/outputs/baseline_action_score.csv
Rows: 30000
   rank  baseline_action_score               reason_code
0     1               1.918245  HIGH_DEMAND_LOW_SESSIONS
1     2               1.918009  HIGH_DEMAND_LOW_SESSIONS
2     3               1.917718  HIGH_DEMAND_LOW_SESSIONS
3     4               1.917718  HIGH_DEMAND_LOW_SESSIONS
4     5               1.917173  HIGH_DEMAND_LOW_SESSIONS
5     6               1.917173  HIGH_DEMAND_LOW_SESSIONS
6     7               1.917173  HIGH_DEMAND_LOW_SESSIONS
7     8               1.917173  HIGH_DEMAND_LOW_SESSIONS
8     9               1.915666  HIGH_DEMAND_LOW_SESSIONS
9    10               1.915666  HIGH_DEMAND_LOW_SESSIONS


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I would review the top 20 pages first. The action is to investigate these pages for content refresh or other improvement opportunities. Confidence is limited because this is a simple baseline rule. A pick could be wrong if search demand does not translate into useful traffic, if the page has other constraints not represented in the data, or if the measured signals are incomplete.

In [18]:
# Top-20 review queue

top20 = df.head(20).copy()

top20["action"] = "REVIEW"
top20["confidence_note"] = (
    "Directional baseline; manual review required."
)
top20["what_would_make_it_wrong"] = (
    "Demand may not translate into useful traffic or the data may be incomplete."
)

print(
    top20[[
        "rank",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]].to_string(index=False)
)

 rank action              reason_code                               confidence_note                                                    what_would_make_it_wrong
    1 REVIEW HIGH_DEMAND_LOW_SESSIONS Directional baseline; manual review required. Demand may not translate into useful traffic or the data may be incomplete.
    2 REVIEW HIGH_DEMAND_LOW_SESSIONS Directional baseline; manual review required. Demand may not translate into useful traffic or the data may be incomplete.
    3 REVIEW HIGH_DEMAND_LOW_SESSIONS Directional baseline; manual review required. Demand may not translate into useful traffic or the data may be incomplete.
    4 REVIEW HIGH_DEMAND_LOW_SESSIONS Directional baseline; manual review required. Demand may not translate into useful traffic or the data may be incomplete.
    5 REVIEW HIGH_DEMAND_LOW_SESSIONS Directional baseline; manual review required. Demand may not translate into useful traffic or the data may be incomplete.
    6 REVIEW HIGH_DEMAND_LOW_SESSIONS Di

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some picks may be weak because the rule only uses a small number of measured signals. A page can have high search demand and low sessions for legitimate reasons, so the score should not be treated as proof that a refresh will improve performance. I also check that future outcome fields and product flags are not being used by the baseline.

In [19]:
# Weak-pick and leakage checks

print("Lowest-scoring top-20 item:")
print(top20.tail(1)[[
    "rank",
    "baseline_action_score",
    "reason_code"
]].to_string(index=False))

future_or_label = [
    c for c in df.columns
    if any(x in c.lower() for x in [
        "label",
        "target",
        "outcome",
        "90d"
    ])
]

product_flags = [
    c for c in df.columns
    if "product" in c.lower()
]

print("\nPotential future/label fields found:")
print(future_or_label)

print("\nPotential product-flag fields found:")
print(product_flags)

print("\nBaseline features used:")
print(["search_volume", "sessions_90d"])

Lowest-scoring top-20 item:
 rank  baseline_action_score              reason_code
   20               1.912797 HIGH_DEMAND_LOW_SESSIONS

Potential future/label fields found:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']

Potential product-flag fields found:
[]

Baseline features used:
['search_volume', 'sessions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.